# 02 — Trade capture & pricing (demo)

Synthetic OTC-style deals with REMIT-shaped fields (LEI, clearing mode). **Not** real trades.

See `features/02-trade-capture-and-pricing.md`.


In [ ]:
import os
import sys
from pathlib import Path

# Uploaded demo modules (Databricks). `dbfs:/tmp/...` is readable on shared UC clusters; FileStore path is a fallback.
sys.path.insert(0, "/dbfs/tmp/energy-trading-forecast-demo")
sys.path.insert(0, "/dbfs/FileStore/energy-trading-forecasting-demo/demo_data")
_env = os.environ.get("DEMO_DATA_PATH", "").strip()
if _env:
    sys.path.insert(0, _env)

CWD = Path.cwd()
for dd in (
    CWD / "modules" / "forecasting" / "demo_data",
    CWD / "demo_data",
    CWD.parent / "demo_data",
):
    if (dd / "notebook_helpers.py").is_file():
        sys.path.insert(0, str(dd))
        break

import notebook_helpers as nh
nh.ensure_demo_data_on_path()
# Matches modules/forecasting/README.md → unity_catalog.schemas (override with DEMO_UC_* env)
print(
    "Unity Catalog Delta target:",
    nh.catalog_schema(),
    "— example:",
    nh.full_table_name("demo_prices_spot_hourly"),
)

import european_demo_sources as eds
import synthetic_generators as sg

spark = nh.get_spark()


In [ ]:

nh.write_demo_table(
    "demo_trades_otc_remit_style",
    eds.otc_trades_remit_style_rows(120),
    eds.OTC_TRADES_COLUMNS,
    spark=spark,
)
# Supporting reference + downstream registry (catalogue style)
nh.write_demo_table(
    "demo_consumer_contracts",
    sg.consumer_contracts_rows(),
    ("consumer_id", "contract_version", "status", "notes"),
    spark=spark,
)
nh.write_demo_table(
    "demo_downstream_exports",
    sg.downstream_export_rows(),
    ("export_id", "consumer", "forecast_scope_id", "exported_at", "status", "table_ref", "schema_version"),
    spark=spark,
)
print("Trade capture demo tables written.")
